In [1]:
import os, shutil, json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = Path("/content/ML-Final-Lab-Group-05")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
SRC_DIR = PROJECT_DIR / "src"

for d in [RAW_DIR, PROCESSED_DIR, SRC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Project folders ready.")

Project folders ready.


In [2]:
uploaded = files.upload()   # select credit_card_fraud_raw_reduced_v3.csv (150 fraud / 88,235 total, 0.17% ratio)
raw_filename = list(uploaded.keys())[0]

# Keep a pristine, untouched copy — this is your source-of-truth
shutil.copy(raw_filename, RAW_DIR / "credit_card_fraud_raw.csv")

df = pd.read_csv(raw_filename)
print("Raw shape:", df.shape)
df.head()

Saving credit_card_fraud_raw_reduced_v3.csv to credit_card_fraud_raw_reduced_v3.csv
Raw shape: (88235, 15)


,trans_date_trans_time,merchant,category,amt,city,state,lat,long,city_pop,job,dob,trans_num,merch_lat,merch_long,is_fraud
0,2020-06-06 17:20:46,"Hoppe, Harris and Bedn",entertainment,12.85,Hawthorne,CA,33.9143,-118.3493,93193,"Editor, magazine features",1995-04-19,e730f1935a095deebe33994b12f17177,34.436244,-118.228511,0
1,2020-03-18 23:30:36,"Gottlieb, Considine and Schultz",shopping_net,2.12,Rock Springs,WY,41.6060,-109.2300,27971,Music therapist,1984-08-01,2cc6e6e3b0d409dee88aaee205e69b32,41.960734,-109.673451,0
2,2020-09-27 18:07:19,Bins-Tillman,entertainment,68.19,Helm,CA,36.4992,-120.0936,123,Early years teacher,1973-02-07,28bcc1336f61e190c8e72ce444e6d174,36.669117,-120.405206,0
3,2020-03-31 03:44:43,Heidenreich PLC,grocery_pos,53.99,Tekoa,WA,47.2271,-117.0819,895,Clothing/textile technologist,1999-05-31,21f54cfaef6f611a433cd8d683067c02,47.369144,-117.165081,0
4,2019-11-03 21:27:37,Predovic Inc,shopping_net,123.61,Lakeport,CA,39.0470,-122.9328,11256,Podiatrist,1972-10-18,7c96acccc4fb673c533b304e868452e5,39.405640,-122.747569,0


In [3]:
TARGET = "is_fraud"

missing = df.isnull().sum()
duplicates = df.duplicated().sum()

print("Missing values per column:\n", missing[missing > 0] if missing.sum() > 0 else "None found")
print("\nDuplicate rows:", duplicates)
print("\nTarget distribution:\n", df[TARGET].value_counts())

# Handle what's found (safe even if counts are 0)
df = df.drop_duplicates()
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ["float64", "int64"]:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0])

# Save this audit as your required "data quality log"
audit = pd.DataFrame({
    "missing_values": missing,
    "dtype": df.dtypes.astype(str)
})
audit.to_csv(PROCESSED_DIR / "data_quality_audit.csv")
print("\nShape after cleaning:", df.shape)

Missing values per column:
 None found

Duplicate rows: 0

Target distribution:
 is_fraud
0    88085
1      150
Name: count, dtype: int64

Shape after cleaning: (88235, 15)


In [4]:
cols_to_drop = ["merchant", "trans_num"]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)
print("Dropped:", cols_to_drop)
print("Remaining columns:", df.columns.tolist())

Dropped: ['merchant', 'trans_num']
Remaining columns: ['trans_date_trans_time', 'category', 'amt', 'city', 'state', 'lat', 'long', 'city_pop', 'job', 'dob', 'merch_lat', 'merch_long', 'is_fraud']


In [5]:
# Age from dob — calculated AT TRANSACTION TIME (not "today"), so the
# feature reflects the customer's age when the transaction actually occurred.
df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")

df["age"] = (
    df["trans_date_trans_time"].dt.year
    - df["dob"].dt.year
    - (
        (df["trans_date_trans_time"].dt.month < df["dob"].dt.month)
        | (
            (df["trans_date_trans_time"].dt.month == df["dob"].dt.month)
            & (df["trans_date_trans_time"].dt.day < df["dob"].dt.day)
        )
    ).astype(int)
)

# Hour and day-of-week from transaction timestamp
df["trans_hour"] = df["trans_date_trans_time"].dt.hour
df["trans_dayofweek"] = df["trans_date_trans_time"].dt.dayofweek

# Distance between customer and merchant (Haversine)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df["lat"], df["long"], df["merch_lat"], df["merch_long"])

# Drop the raw columns now that we've extracted what we need from them
df = df.drop(columns=["dob", "trans_date_trans_time"])

print("Shape after feature engineering:", df.shape)
df.head()

Shape after feature engineering: (88235, 15)


,category,amt,city,state,lat,long,city_pop,job,merch_lat,merch_long,is_fraud,age,trans_hour,trans_dayofweek,distance_km
0,entertainment,12.85,Hawthorne,CA,33.9143,-118.3493,93193,"Editor, magazine features",34.436244,-118.228511,0,25,17,5,59.091675
1,shopping_net,2.12,Rock Springs,WY,41.6060,-109.2300,27971,Music therapist,41.960734,-109.673451,0,35,23,2,53.923925
2,entertainment,68.19,Helm,CA,36.4992,-120.0936,123,Early years teacher,36.669117,-120.405206,0,47,18,6,33.631403
3,grocery_pos,53.99,Tekoa,WA,47.2271,-117.0819,895,Clothing/textile technologist,47.369144,-117.165081,0,20,3,1,16.994574
4,shopping_net,123.61,Lakeport,CA,39.0470,-122.9328,11256,Podiatrist,39.405640,-122.747569,0,47,21,6,42.952314


In [6]:
for col in ["amt", "age", "distance_km", "city_pop"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)]
    print(f"{col}: {len(outliers)} potential outliers (kept — real fraud cases often are outliers)")

amt: 4543 potential outliers (kept — real fraud cases often are outliers)
age: 0 potential outliers (kept — real fraud cases often are outliers)
distance_km: 0 potential outliers (kept — real fraud cases often are outliers)
city_pop: 14328 potential outliers (kept — real fraud cases often are outliers)


In [7]:
cleaned_path = PROCESSED_DIR / "cleaned_fraud_data.csv"
df.to_csv(cleaned_path, index=False)
print("Saved:", cleaned_path, "| Shape:", df.shape)

Saved: /content/ML-Final-Lab-Group-05/data/processed/cleaned_fraud_data.csv | Shape: (88235, 15)


In [8]:
check = pd.read_csv(cleaned_path)
print(check.shape)
check.head(10)

(88235, 15)


,category,amt,city,state,lat,long,city_pop,job,merch_lat,merch_long,is_fraud,age,trans_hour,trans_dayofweek,distance_km
0,entertainment,12.85,Hawthorne,CA,33.9143,-118.3493,93193,"Editor, magazine features",34.436244,-118.228511,0,25,17,5,59.091675
1,shopping_net,2.12,Rock Springs,WY,41.6060,-109.2300,27971,Music therapist,41.960734,-109.673451,0,35,23,2,53.923925
2,entertainment,68.19,Helm,CA,36.4992,-120.0936,123,Early years teacher,36.669117,-120.405206,0,47,18,6,33.631403
3,grocery_pos,53.99,Tekoa,WA,47.2271,-117.0819,895,Clothing/textile technologist,47.369144,-117.165081,0,20,3,1,16.994574
4,shopping_net,123.61,Lakeport,CA,39.0470,-122.9328,11256,Podiatrist,39.405640,-122.747569,0,47,21,6,42.952314
5,shopping_pos,2.12,Unionville,MO,40.4815,-92.9951,3805,"Investment banker, corporate",40.458693,-92.918307,0,70,1,1,6.973474
6,grocery_pos,221.87,Kansas City,MO,38.9621,-94.5959,545147,Counsellor,39.430692,-94.988709,0,32,7,4,62.134864
7,misc_pos,8.26,Burlington,WA,48.4786,-122.3345,14871,Public house manager,47.518157,-121.732104,0,46,5,2,115.819914
8,gas_transport,69.81,Freedom,WY,43.0172,-111.0292,471,"Education officer, museum",42.599648,-111.302823,0,52,3,3,51.516354
9,gas_transport,49.60,Matthews,MO,36.7154,-89.6287,1019,Aeronautical engineer,35.928269,-88.720002,0,40,10,4,119.532150


In [9]:
X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print(y_train.value_counts(), "\n", y_test.value_counts())

Train: (70588, 14) | Test: (17647, 14)
is_fraud
0    70468
1      120
Name: count, dtype: int64 
 is_fraud
0    17617
1       30
Name: count, dtype: int64


In [10]:
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), categorical_cols)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("Train processed:", X_train_processed.shape)
print("Test processed:", X_test_processed.shape)

Train processed: (70588, 356)
Test processed: (17647, 356)


In [11]:
np.savez(PROCESSED_DIR / "X_train.npz", X=X_train_processed)
np.savez(PROCESSED_DIR / "X_test.npz", X=X_test_processed)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

with open(PROCESSED_DIR / "feature_names.json", "w") as f:
    json.dump(list(feature_names), f)

joblib.dump(preprocessor, PROCESSED_DIR / "preprocessor.joblib")

print("All artifacts saved to:", PROCESSED_DIR)
print(os.listdir(PROCESSED_DIR))

All artifacts saved to: /content/ML-Final-Lab-Group-05/data/processed
['y_test.csv', 'X_test.npz', 'data_quality_audit.csv', 'cleaned_fraud_data.csv', 'preprocessor.joblib', 'X_train.npz', 'feature_names.json', 'y_train.csv']


In [12]:
preprocessing_script = '''
import pandas as pd
import numpy as np

def clean_and_engineer(df):
    df = df.drop_duplicates()
    df = df.drop(columns=[c for c in ["merchant", "trans_num"] if c in df.columns])

    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")

    # Age calculated AT TRANSACTION TIME (not "today"), so the feature
    # reflects the customer's age when the transaction actually occurred.
    df["age"] = (
        df["trans_date_trans_time"].dt.year
        - df["dob"].dt.year
        - (
            (df["trans_date_trans_time"].dt.month < df["dob"].dt.month)
            | (
                (df["trans_date_trans_time"].dt.month == df["dob"].dt.month)
                & (df["trans_date_trans_time"].dt.day < df["dob"].dt.day)
            )
        ).astype(int)
    )

    df["trans_hour"] = df["trans_date_trans_time"].dt.hour
    df["trans_dayofweek"] = df["trans_date_trans_time"].dt.dayofweek

    R = 6371
    lat1, lon1 = np.radians(df["lat"]), np.radians(df["long"])
    lat2, lon2 = np.radians(df["merch_lat"]), np.radians(df["merch_long"])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    df["distance_km"] = R * 2 * np.arcsin(np.sqrt(a))

    df = df.drop(columns=["dob", "trans_date_trans_time"])
    return df
'''

with open(SRC_DIR / "preprocessing.py", "w") as f:
    f.write(preprocessing_script)

print("Saved src/preprocessing.py")

Saved src/preprocessing.py


In [13]:
zip_path = shutil.make_archive("/content/ML-Group05-DE-artifacts", "zip", PROJECT_DIR)
print("Created:", zip_path)

files.download(zip_path)          # single download — all artifacts inside
files.download(str(cleaned_path)) # the processed dataset CSV, separately, since you need it standalone

Created: /content/ML-Group05-DE-artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>